# Áudios da Jornada com São Miguel (OmniVoice no Colab)

Gera os áudios da jornada com voz clonada, sem instalar nada no seu computador.
Tudo fica salvo no seu Google Drive, então dá para parar e continuar depois.

**Antes de começar**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.
2. No Google Drive, crie a pasta **`JDA-audio`** e coloque nela:
   - `voz-referencia.wav`: **8 a 15 segundos** da voz, em ambiente silencioso, no tom calmo que você quer na narração. Só use a voz de alguém que autorizou.
   - `canto.mp3` (opcional por enquanto): o Canto de São Miguel gerado no Suno.
   - `fundo.mp3` (opcional): trilha de fundo suave para a narração.
3. Rode as células na ordem. Na primeira vez, a instalação e o download do modelo levam alguns minutos.

**Saída:** `JDA-audio/trabalho/saida/d01.mp3`, `d02.mp3`… prontos para subir no R2.

In [ ]:
#@title 1. Configuração
PASTA_DRIVE = "/content/drive/MyDrive/JDA-audio"  #@param {type:"string"}
VOZ_REFERENCIA = "voz-referencia.wav"  #@param {type:"string"}
#@markdown Transcrição **exata** do que é dito no áudio de referência. Melhora a fidelidade da voz. Se deixar vazio, o Whisper transcreve sozinho.
TRANSCRICAO_REFERENCIA = ""  #@param {type:"string"}
#@markdown Velocidade da fala (1.0 = padrão do modelo; abaixo de 1 = mais calmo).
VELOCIDADE = 0.95  #@param {type:"slider", min:0.8, max:1.1, step:0.05}
#@markdown Passos de geração: 32 = melhor qualidade, 16 = mais rápido.
PASSOS = 32  #@param [16, 32] {type:"raw"}
CANTO = "canto.mp3"  #@param {type:"string"}
FUNDO = "fundo.mp3"  #@param {type:"string"}
REPO = "https://github.com/lnz1337/JDA.git"

In [ ]:
#@title 2. Preparar ambiente (GPU, Drive, repositório, OmniVoice)
import os, subprocess, sys, importlib
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "NENHUMA (troque o ambiente para GPU T4; em CPU fica bem mais lento)")

from google.colab import drive
drive.mount("/content/drive")
pasta = Path(PASTA_DRIVE)
pasta.mkdir(parents=True, exist_ok=True)

ref = pasta / VOZ_REFERENCIA
if not ref.exists():
    raise SystemExit(f"Coloque o áudio de referência em {ref} e rode de novo.")
canto = pasta / CANTO if CANTO and (pasta / CANTO).exists() else None
fundo = pasta / FUNDO if FUNDO and (pasta / FUNDO).exists() else None
print("Referência:", ref)
print("Canto:", canto or "não encontrado (vai usar 3 s de silêncio no lugar)")
print("Fundo:", fundo or "sem trilha de fundo")

# Repositório: clona (ou atualiza). Se o repo for privado, crie o segredo GITHUB_TOKEN
# no ícone de chave da barra lateral do Colab.
repo_dir = Path("/content/JDA")
url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(repo_dir)], check=True)
print("Repositório:", subprocess.run(["git", "-C", str(repo_dir), "log", "--oneline", "-1"],
                                     capture_output=True, text=True).stdout.strip())

try:
    import omnivoice  # noqa: F401
except ImportError:
    print("Instalando OmniVoice (só na primeira vez da sessão)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omnivoice"], check=True)

os.environ["JDA_AUDIO_DIR"] = str(pasta / "trabalho")
os.environ["OMNIVOICE_REF_AUDIO"] = str(ref)
os.environ["OMNIVOICE_REF_TEXT"] = TRANSCRICAO_REFERENCIA.strip()
os.environ["OMNIVOICE_SPEED"] = str(VELOCIDADE)
os.environ["OMNIVOICE_STEPS"] = str(PASSOS)

sys.path.insert(0, str(repo_dir / "conteudo"))
import gerar_seed, gerar_audio as ga
importlib.reload(gerar_seed); importlib.reload(ga)
dias = {d["day"]: d for d in gerar_seed.carregar_dias()}
prontos = [n for n, d in dias.items() if d["audio_script"]]
print("Roteiros prontos:", prontos)

In [ ]:
#@title 3. Teste rápido da voz (ouça antes de gerar os dias)
from IPython.display import Audio, display
TEXTO_TESTE = "Em nome do Pai, do Filho e do Espírito Santo. Amém. Seja bem-vindo, seja bem-vinda ao primeiro dia da sua jornada com São Miguel Arcanjo."  #@param {type:"string"}

saida_teste = pasta / "teste.wav"
ga.sintetizar_omnivoice(TEXTO_TESTE, saida_teste)
display(Audio(str(saida_teste)))
print("Não gostou? Ajuste VELOCIDADE, troque o áudio de referência ou preencha a transcrição na célula 1, rode a 2 e teste de novo.")

In [ ]:
#@title 4. Gerar os áudios dos dias
#@markdown Ex.: `1` para um dia, `1-7` para um intervalo, `1,3,5` para vários.
DIAS = "1"  #@param {type:"string"}
#@markdown Marque se você mudou o texto de um roteiro: regera todos os trechos dos dias escolhidos.
REFAZER_TUDO = False  #@param {type:"boolean"}

def interpretar(spec):
    numeros = set()
    for parte in spec.replace(" ", "").split(","):
        if "-" in parte:
            a, b = map(int, parte.split("-"))
            numeros.update(range(a, b + 1))
        elif parte:
            numeros.add(int(parte))
    return sorted(numeros)

for n in interpretar(DIAS):
    dia = dias.get(n)
    if not dia or not dia["audio_script"]:
        print(f"Dia {n}: roteiro ainda não escrito, pulando.")
        continue
    ga.cmd_planejar(dia)
    ga.cmd_sintetizar(dia, "omnivoice", REFAZER_TUDO)
    ga.cmd_montar(dia, canto, fundo)
    mp3 = ga.PASTA_AUDIO / "saida" / f"d{n:02d}.mp3"
    print(f"Ouça o dia {n}:")
    display(Audio(str(mp3)))

In [ ]:
#@title 5. Refazer um trecho específico (quando uma frase saiu ruim)
#@markdown Os trechos ficam em `JDA-audio/trabalho/dNN/trechos/NNN.txt`. Informe o dia e o número do trecho.
DIA_TRECHO = 1  #@param {type:"integer"}
TRECHO = 1  #@param {type:"integer"}

pasta_trechos = ga.PASTA_AUDIO / f"d{DIA_TRECHO:02d}" / "trechos"
txt = pasta_trechos / f"{TRECHO:03d}.txt"
wav = pasta_trechos / f"{TRECHO:03d}.wav"
if not txt.exists():
    raise SystemExit(f"Trecho {txt} não existe. Gere o dia na célula 4 primeiro.")
print("Texto:", txt.read_text(encoding="utf-8").strip())
wav.unlink(missing_ok=True)
ga.cmd_sintetizar(dias[DIA_TRECHO], "omnivoice", False)
display(Audio(str(wav)))
ga.cmd_montar(dias[DIA_TRECHO], canto, fundo)
display(Audio(str(ga.PASTA_AUDIO / "saida" / f"d{DIA_TRECHO:02d}.mp3")))